# Profiles to HEALPix

Convert EarthCARE atmospheric profile data (ATLID lidar, CPR radar) to HEALPix:

- 1D lat/lon along the ground track
- A vertical dimension (height) with atmospheric columns
- Each profile is mapped to a HEALPix cell, the vertical axis is preserved

In [1]:
import earthcarekit as eck
import healpix_geo as hpxg
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xdggs
import zarr

import cartopy.crs as ccrs
import cartopy.feature as cfeature


from earthcare_dggs.convert import lonlat_to_healpix_cells
from earthcare_dggs.settings import ATLID_DEPTH, ELLIPSOID

## Input parameters

In [2]:
PRODUCT = "ACM_CAP_2B"
ORBIT = "01164D"
OUTPUT_PATH = "/work/ks1387/gw/data/obs/earthcare/"

## Load product

In [3]:
result = eck.search_product(file_type=PRODUCT, orbit_and_frame=ORBIT)
with eck.read_product(result.filepath[0]) as ds:
    profile = ds.load()

print(f"Dimensions: {dict(profile.sizes)}")
print(f"Lat: [{float(profile['latitude'].min()):.1f}, {float(profile['latitude'].max()):.1f}]")
print(f"Lon: [{float(profile['longitude'].min()):.1f}, {float(profile['longitude'].max()):.1f}]")
print(f"\n2D variables (along_track, vertical):")
for v in profile.data_vars:
    if profile[v].dims == ("along_track", "vertical"):
        print(f"  {v}: shape={profile[v].shape}")

Dimensions: {'along_track': 4950, 'vertical': 242, 'MSI_longwave_channel': 3, 'MSI_shortwave_channel': 1}
Lat: [22.5, 67.5]
Lon: [-69.9, -52.7]

2D variables (along_track, vertical):
  height: shape=(4950, 242)
  ice_extinction: shape=(4950, 242)
  ice_extinction_error: shape=(4950, 242)
  ice_extinction_error_corr_scale: shape=(4950, 242)
  ice_extinction_kernel_sum: shape=(4950, 242)
  ice_extinction_kernel_corr_scale: shape=(4950, 242)
  ice_N0prime: shape=(4950, 242)
  ice_N0prime_error: shape=(4950, 242)
  ice_N0prime_error_corr_scale: shape=(4950, 242)
  ice_N0prime_kernel_sum: shape=(4950, 242)
  ice_N0prime_kernel_corr_scale: shape=(4950, 242)
  ice_lidar_bscat_extinction_ratio: shape=(4950, 242)
  ice_lidar_bscat_extinction_ratio_error: shape=(4950, 242)
  ice_lidar_bscat_extinction_ratio_error_corr_scale: shape=(4950, 242)
  ice_lidar_bscat_extinction_ratio_kernel_sum: shape=(4950, 242)
  ice_lidar_bscat_extinction_ratio_kernel_corr_scale: shape=(4950, 242)
  ice_riming_index

## Assign HEALPix cell IDs

For profile data, we assign each along-track position to its HEALPix cell
and add `cell_ids` as a coordinate. The vertical dimension is preserved as-is.

In [4]:
prof_cell_ids = lonlat_to_healpix_cells(
    profile["longitude"].values, profile["latitude"].values,
    depth=ATLID_DEPTH, ellipsoid=ELLIPSOID,
)

unique_cells = np.unique(prof_cell_ids)
print(f"HEALPix depth {ATLID_DEPTH}: {len(profile.along_track)} profiles → {len(unique_cells)} unique cells")
print(f"Average profiles per cell: {len(profile.along_track) / len(unique_cells):.1f}")

# Add cell_ids as coordinate
profile_healpix = profile.assign_coords(cell_ids=("along_track", prof_cell_ids))
profile_healpix.attrs.update({
    "healpix_level": ATLID_DEPTH,
    "healpix_indexing": "nested",
    "healpix_ellipsoid": ELLIPSOID,
})

print(f"\nDataset with cell_ids coordinate:")
print(profile_healpix)

HEALPix depth 14: 4950 profiles → 4950 unique cells
Average profiles per cell: 1.0

Dataset with cell_ids coordinate:
<xarray.Dataset> Size: 561MB
Dimensions:                                                         (
                                                                     along_track: 4950,
                                                                     vertical: 242,
                                                                     MSI_longwave_channel: 3,
                                                                     MSI_shortwave_channel: 1)
Coordinates:
    cell_ids                                                        (along_track) uint64 40kB ...
Dimensions without coordinates: along_track, vertical, MSI_longwave_channel,
                                MSI_shortwave_channel
Data variables: (12/201)
    filename                                                        <U60 240B ...
    file_type                                                       <U10 

In [5]:
profile_healpix

<xarray.Dataset> Size: 561MB
Dimensions:                                                         (
                                                                     along_track: 4950,
                                                                     vertical: 242,
                                                                     MSI_longwave_channel: 3,
                                                                     MSI_shortwave_channel: 1)
Coordinates:
    cell_ids                                                        (along_track) uint64 40kB ...
Dimensions without coordinates: along_track, vertical, MSI_longwave_channel,
                                MSI_shortwave_channel
Data variables: (12/201)
    filename                                                        <U60 240B ...
    file_type                                                       <U10 40B ...
    frame_id                                                        <U1 4B 'D'
    orbit_number                                                    int64 8B ...
    orbit_and_frame                                                 <U6 24B '...
    baseline                                                        <U2 8B 'BA'
    ...                                                              ...
    MSI_shortwave_observation_variable_count                        (along_track) int32 20kB ...
    MSI_shortwave_cost_function                                     (along_track) float32 20kB ...
    MSI_shortwave_wavelength                                        (MSI_shortwave_channel) float32 4B ...
    MSI_shortwave_albedo_forward                                    (along_track, MSI_shortwave_channel) float32 20kB ...
    MSI_shortwave_albedo                                            (along_track, MSI_shortwave_channel) float32 20kB ...
    MSI_shortwave_albedo_assimilation_status                        (along_track) int8 5kB ...
Attributes:
    healpix_level:      14
    healpix_indexing:   nested
    healpix_ellipsoid:  WGS84

## Save to DGGS ZARR

In [6]:
# Add metadata
profile_healpix.attrs.update({
    "source": "EarthCARE MSI L2A",
    "healpix_level": ATLID_DEPTH,
    "healpix_indexing": "nested",
    "healpix_ellipsoid": ELLIPSOID,
    "orbit_and_frame": ORBIT,
})

# Add xdggs index
profile_healpix = xdggs.decode(
    profile_healpix,
    grid_info=xdggs.HealpixInfo(level=ATLID_DEPTH, indexing_scheme="nested"),
)

# Encode with xdggs convention
profile_encoded = xdggs.encode(profile_healpix, "xdggs")

# Write to Zarr
output_path = f"{OUTPUT_PATH}earthcare_ACM_CAP_2B_{ORBIT}.zarr"
profile_encoded.to_zarr(output_path, mode="w", consolidated=True)

# Add DGGS Zarr convention metadata (compatible with legacy-converters format)
dggs_convention = {
    "uuid": "7b255807-140c-42ca-97f6-7a1cfecdbc38",
    "name": "dggs",
    "schema_url": "https://raw.githubusercontent.com/zarr-conventions/dggs/refs/tags/v1/schema.json",
    "spec_url": "https://github.com/zarr-conventions/dggs/blob/v1/README.md",
    "description": "Discrete Global Grid Systems convention for zarr",
}

dggs_meta = {
    "name": "healpix",
    "refinement_level": ATLID_DEPTH,
    "indexing_scheme": "nested",
    "ellipsoid": {
        "name": "wgs84",
        "semimajor_axis": 6378137.0,
        "inverse_flattening": 298.257223563,
    },
    "spatial_dimension": "cell_ids",
    "coordinate": "cell_ids",
    "compression": "none",
}

root = zarr.open_group(output_path, mode="r+")
root.attrs["zarr_conventions"] = [dggs_convention]
root.attrs["dggs"] = dggs_meta

print(f"Saved to {output_path}")
print(f"\nDGGS metadata: {dggs_meta}")

/home/k/k204228/.conda/envs/earthcare/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=60, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/k/k204228/.conda/envs/earthcare/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=1, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future ver

Saved to /work/ks1387/gw/data/obs/earthcare/earthcare_ACM_CAP_2B_01164D.zarr

DGGS metadata: {'name': 'healpix', 'refinement_level': 14, 'indexing_scheme': 'nested', 'ellipsoid': {'name': 'wgs84', 'semimajor_axis': 6378137.0, 'inverse_flattening': 298.257223563}, 'spatial_dimension': 'cell_ids', 'coordinate': 'cell_ids', 'compression': 'none'}


/home/k/k204228/.conda/envs/earthcare/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
